In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/application_train.csv")
print(df.shape)

(307511, 122)


In [5]:
df.info()
print(df['TARGET'].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), object(16)
memory usage: 286.2+ MB
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [6]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df['SK_ID_CURR'].duplicated().sum())

Duplicate rows: 0
Duplicate IDs: 0


In [7]:
missing = (df.isnull().mean() * 100).sort_values(ascending=False)
missing[missing > 0].head(20)

,0
COMMONAREA_AVG,69.872297
COMMONAREA_MODE,69.872297
COMMONAREA_MEDI,69.872297
NONLIVINGAPARTMENTS_MEDI,69.432963
NONLIVINGAPARTMENTS_MODE,69.432963
NONLIVINGAPARTMENTS_AVG,69.432963
FONDKAPREMONT_MODE,68.386172
LIVINGAPARTMENTS_AVG,68.354953
LIVINGAPARTMENTS_MEDI,68.354953
LIVINGAPARTMENTS_MODE,68.354953


In [8]:
print("365243 rows:", (df['DAYS_EMPLOYED'] == 365243).sum())
df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

print("XNA gender:", (df['CODE_GENDER'] == 'XNA').sum())
df = df[df['CODE_GENDER'] != 'XNA']
df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].replace('XNA', np.nan)

365243 rows: 55374
XNA gender: 4


In [9]:
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Not Provided')

bureau_cols = [c for c in df.columns if c.startswith('AMT_REQ_CREDIT_BUREAU')]
df[bureau_cols] = df[bureau_cols].fillna(0)

for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    df[c + '_MISSING'] = df[c].isnull().astype(int)
    df[c] = df[c].fillna(df[c].median())

df['OWN_CAR_AGE'] = df['OWN_CAR_AGE'].fillna(0)

building_cols = [c for c in df.columns if c.endswith(('_AVG', '_MODE', '_MEDI'))]
df = df.drop(columns=building_cols)

num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

In [10]:
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).astype(int)
df['AgeBucket'] = pd.cut(df['AGE_YEARS'], bins=[0, 25, 35, 50, 65, 100],
                         labels=['Young', 'Early Career', 'Mid Career', 'Established', 'Senior'])
df['Income_Level'] = pd.qcut(df['AMT_INCOME_TOTAL'], q=3, labels=['Low', 'Medium', 'High'])
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['LOAN_TERM_MONTHS'] = (df['AMT_CREDIT'] / df['AMT_ANNUITY']).round()

In [11]:
print(df.shape, "| Missing left:", df.isnull().sum().sum())

from google.colab import files
df.to_csv("cleaned_data.csv", index=False)
files.download("cleaned_data.csv")

(307507, 85) | Missing left: 56666


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>